# 🛰️ SVAMITVA Unified DGX Pipeline (V3)
### Developed by Students of Digital University Kerala (DUK)

This notebook provides a professional entry point for training the full SVAMITVA feature extraction suite on DGX systems.

### 🎯 Pipeline Architecture:
1. **Stage 1: YOLO Prep** — Automatic raster-to-YOLO conversion for point features.
2. **Stage 2: Segmentation** — Multi-task Ensemble training (SAM2 backbone).
3. **Stage 3: YOLO Train** — High-precision utility localization.

**Target Accuracy:** ≥95% IoU.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"
import sys
import torch
import logging
from pathlib import Path
import importlib
import shutil

# ── Environment Setup ──
MANUAL_REPO_ROOT = None

def find_repo_root(start_path, markers=["train.py", "models"], max_depth=5):
    curr = Path(start_path).resolve()
    for _ in range(max_depth):
        for marker in markers:
            if (curr / marker).exists():
                return curr
        curr = curr.parent
    return Path(start_path).resolve()

if MANUAL_REPO_ROOT:
    REPO_ROOT = Path(MANUAL_REPO_ROOT).resolve()
else:
    try:
        REPO_ROOT = find_repo_root(os.getcwd())
    except:
        REPO_ROOT = Path(".").resolve()

os.chdir(str(REPO_ROOT))

repo_path_str = str(REPO_ROOT)
if repo_path_str not in sys.path:
    sys.path.insert(0, repo_path_str)
importlib.invalidate_caches()

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")
logging.getLogger("train_engine").setLevel(logging.INFO)

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"GPUs: {torch.cuda.device_count()}")

## 🧠 Unified Training (Recommended)

The most efficient way to train is via the unified `train.py` orchestrator. This handles all three stages of the pipeline in sequence.

```bash
python train.py --train_dirs ./data/MAP1 --epochs 150 --cache_features
```


In [ ]:
# Configuration cell for manual training tweak if needed
from train_engine.config import TrainingConfig

RUN_NAME = "final_hackathon_submission"

config = TrainingConfig(
    train_dirs=[Path("data/MAP1")],
    batch_size=32,
    num_epochs=150,
    learning_rate=3e-4,
    mixed_precision=True,
    freeze_backbone_epochs=5,
    checkpoint_dir=Path(f"checkpoints/{RUN_NAME}"),
    num_workers=8,
    sam2_checkpoint=REPO_ROOT / "checkpoints" / "sam2.1_hiera_tiny.pt"
)

print(f"Target Checkpoint: {config.checkpoint_dir.absolute()}")

## 🚀 Start Automated Training

Run the full pipeline directly from this notebook:

In [ ]:
!python train.py --train_dirs ./data/MAP1 --epochs 150 --cache_features

---
## 📊 Multi-GPU DDP (Terminal Only)

To fully leverage all 8 GPUs on the DGX at maximum speed:

```bash
torchrun --nproc_per_node=8 train.py \
    --train_dirs ./data/MAP1 \
    --batch_size 128 --cache_features
```